In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import os
from periodicwave.pbc import lattices

In [ ]:
import pandas as pd

In [ ]:
def convert_moire_scales(me_eff_rel = 0.9, eps_inverse = 0.2, moire_a = 9.8, moire_potential_strength = 10.3):
    """ 
    Converts moire system parameters from SI to natural units used in the code.

    Default values as used in M Geier, K Nazaryan, T Zaklama, and L Fu, Phys. Rev. B 112, 045119

    Arguments:
        me_eff_rel = 0.35             # effective mass in units of bare electron mass
        moire_a = 8.031               # moire lattice constant, nm
        eps_inverse = 0.1             # inverse relative diel. constant of the surrounding dielectric
        moire_potential_strength = 15 # energy scale V of the hexagonal moire potential, meV 

    Returns:
        energy_scale: Conversion factor between energies returned from the code and SI units (meV)
        V: moire potential strength in natural units
        U: interaction energy scale in natural units
    """
    # Natural constants
    a_0  = 5.29177210544e-2      # Bohr radius, nm
    hbar = 6.582119569509066e-1  # meV * ps
    me   = 5.685630060215049e-3  # electron rest mass, meV/[(nm/ps)^2]

    # effective mass in SI units (meV/[(nm/ps)^2])
    me_eff = me_eff_rel * me      # effective mass, meV/[(nm/ps)^2]

    # scales converting dimensionless units from the code to SI units
    # Here we present units where distances are measured in terms of the moire lattice constant moire_a
    # and energies are measured in \hbar^2 / moire_a^2 / me_eff,
    energy_scale = hbar**2 / moire_a**2 / me_eff # meV
    # In these units, the kinetic energy term is
    # - 0.5 \sum_j \nabla_j^2 where j runs over all electrons.
    # In these units, the dimensionless moire potential strength is V, determining
    # - 2 V \sum_i \sum_{n = 1}^{3} \cos ( g_n \cdot r_i + \phi )
    V = moire_potential_strength / energy_scale 
    # The dimensionless Coulomb interaction energy scale is U, determining
    # 0.5 U \sum_{i \neq j} 1/{|r_j - r_i|}
    U = (moire_a / a_0) * eps_inverse * (me_eff / me)

    return energy_scale, V, U

In [ ]:
def get_positions_from_latest_npz_files(folder_path, N):
    """
    Loads position data from N latest checkpoints of FermiNet ouput stored in
    a folder with path folder_path
    """
    # Find all files that match the "qmcjax_ckpt_XXXXXX.npz" pattern
    files = [
        f for f in os.listdir(folder_path) 
        if f.startswith("qmcjax_ckpt_") and f.endswith(".npz")
    ]
    
    # Extract the six-digit number from each filename and store it in a tuple (number, filename)
    file_numbers = []
    for file in files:
        try:
            number = int(file.split('_')[-1].split('.')[0])
            file_numbers.append((number, file))
        except ValueError:
            print(f"Skipping file {file}, could not extract valid six-digit number.")

    # Sort the files by the six-digit number in descending order
    sorted_files = sorted(file_numbers, key=lambda x: x[0], reverse=True)

    # Select the top N files with the largest numbers
    largest_files = sorted_files[:N]

    print("In folder " + folder_path + "/ loading checkpoints: ")
    print([el[0] for el in largest_files])

    # Load the contents of the selected files
    positions_ckpt = []
    spins_ckpt = []
    for number, filename in largest_files:
        file_path = os.path.join(folder_path, filename)
        try:
            ckpt_data = np.load(file_path, allow_pickle=True)
            print(list(ckpt_data.keys()))
            data = ckpt_data['data'].item()
            print(list(data.keys()))
            positions_ckpt.append(data['positions'])
            spins_ckpt.append(data['spins'])
            # loaded_data[filename] = data
        except Exception as e:
            print(f"Failed to load {filename}: {e}")

    filenames = [el[1] for el in largest_files]

    return positions_ckpt, spins_ckpt, filenames

In [ ]:
def get_positions_from_npz_files(folder_path, N=None, ckpt_range=None, index_range=None,
                                 from_end=True, reverse=True):
    """
    Loads position data from a selected set of FermiNet checkpoints stored in
    a folder with path folder_path.

    N           : number of files to keep (None = keep all that pass the filters)
    ckpt_range  : (lo, hi) inclusive bounds on the checkpoint number; either
                  end may be None, e.g. (200000, None)
    index_range : (i, j) slice on the number-sorted list; negative indices work,
                  e.g. (-20, -10) for the 10 files before the last 10
    from_end    : if N is given, take the N newest (True) or N oldest (False)
    reverse     : return newest first (True) or oldest first (False)
    """
    # Find all files that match the "qmcjax_ckpt_XXXXXX.npz" pattern
    files = [
        f for f in os.listdir(folder_path)
        if f.startswith("qmcjax_ckpt_") and f.endswith(".npz")
    ]

    # Extract the six-digit number from each filename and store it in a tuple (number, filename)
    file_numbers = []
    for file in files:
        try:
            number = int(file.split('_')[-1].split('.')[0])
            file_numbers.append((number, file))
        except ValueError:
            print(f"Skipping file {file}, could not extract valid six-digit number.")

    # Sort the files by the six-digit number in ascending order
    sorted_files = sorted(file_numbers, key=lambda x: x[0])

    # Keep only checkpoints whose number falls inside ckpt_range
    if ckpt_range is not None:
        lo, hi = ckpt_range
        sorted_files = [
            (n, f) for n, f in sorted_files
            if (lo is None or n >= lo) and (hi is None or n <= hi)
        ]

    # Or keep a positional slice of the sorted list
    if index_range is not None:
        i, j = index_range
        sorted_files = sorted_files[i:j]

    # Select N files from whichever end was requested
    if N is not None:
        sorted_files = sorted_files[-N:] if from_end else sorted_files[:N]

    # Order the output
    selected_files = sorted_files[::-1] if reverse else sorted_files

    print("In folder " + folder_path + "/ loading checkpoints: ")
    print([el[0] for el in selected_files])

    # Load the contents of the selected files
    positions_ckpt = []
    spins_ckpt = []
    filenames = []
    for number, filename in selected_files:
        file_path = os.path.join(folder_path, filename)
        try:
            ckpt_data = np.load(file_path, allow_pickle=True)
            print(list(ckpt_data.keys()))
            data = ckpt_data['data'].item()
            print(list(data.keys()))
            positions_ckpt.append(data['positions'])
            spins_ckpt.append(data['spins'])
            filenames.append(filename)
        except Exception as e:
            print(f"Failed to load {filename}: {e}")

    return positions_ckpt, spins_ckpt, filenames

In [ ]:
ndim = 2 # spatial dimension of the system
network_type = "CustomPsiformer"

# system parameters
potential_type = "CoulombMoire"
num_unit_cells = 12
nspins = (18, 18)
num_electrons = sum(nspins)
me_eff_rel = 0.9 # in units of bare electron mass
eps_inverse = 0.2 # inverse dielectric constant of surrounding dielectric
moire_lattice_constant_nm = 9.8 # in nm
moire_potential_strength_meV = 10.3 # in meV
moire_potential_phi = 20 # potential shape angle in degrees

In [ ]:
# Convert SI units to natural units
energy_scale, moire_potential_strength, interaction_energy_scale = convert_moire_scales(me_eff_rel, eps_inverse, moire_lattice_constant_nm, moire_potential_strength_meV)

In [ ]:
# generate folder name
folder_name = f"el{nspins[0]}_{nspins[1]}_N{num_unit_cells}_V{np.round(moire_potential_strength,8)}_{moire_potential_phi}_U{np.round(interaction_energy_scale,8)}"

In [ ]:
moire_a = 1.0
lat_vec, moire_lat_vec, lattice_M = lattices._triangular_lattice_vecs_periodic_potential(a = moire_a, num_sites=num_unit_cells, return_lattice_M = True)
rec = 2 * np.pi * np.linalg.inv(lat_vec)
area = np.abs(np.linalg.det(lat_vec))
r_s = np.sqrt(area / sum(nspins) / np.pi)

In [ ]:
load_N_ckpts = 12 # number of latest checkpoints to load 
batch_size = 2048

In [ ]:
# load configurations from latest checkpoints
positions_ckpt, spins_ckpt, filenames = get_positions_from_latest_npz_files(folder_name2, load_N_ckpts)
#positions_ckpt, spins_ckpt, filenames = get_positions_from_npz_files(folder_name, ckpt_range=(9000, None), N=load_N_ckpts, from_end=False)
shape_pos = np.shape(positions_ckpt)
positions_batch = np.reshape(np.array(positions_ckpt),(shape_pos[0]*shape_pos[1]*shape_pos[2], shape_pos[3]//ndim, ndim))
positions_batch = np.array([lattices.send_positions_to_first_unit_cell(config, lat_vec, rec) for config in positions_batch])


In [ ]:
# plot electron density
fig_n, ax_n = plt.subplots(1, 1, figsize = (7, 5))
positions_plot = np.reshape(np.array(positions_batch),(shape_pos[0]*shape_pos[1]*shape_pos[2]*shape_pos[3]//ndim, ndim))
ax_n.scatter(positions_plot[:,0], positions_plot[:,1], color="tab:blue", s=0.05, alpha=0.2)
ax_n.set_xlabel("x / a_M")
ax_n.set_ylabel("y / a_M")
ax_n.set_title("Electron density")
plt.axis('equal')
plt.show()

In [ ]:
#ndim = 2
#shape_pos = np.shape(positions_ckpt)
nelec = shape_pos[3] // ndim              # 36
n_conf = shape_pos[0] * shape_pos[1] * shape_pos[2]   # 2*4*512 = 4096

positions_batch = np.reshape(np.array(positions_ckpt), (n_conf, nelec, ndim))
positions_batch = np.array([lattices.send_positions_to_first_unit_cell(config, lat_vec, rec)
                            for config in positions_batch])
spins_batch = np.reshape(np.array(spins_ckpt), (n_conf, nelec))

positions_plot = np.reshape(positions_batch, (n_conf * nelec, ndim))
spins_plot     = np.reshape(spins_batch,     (n_conf * nelec,))

print("spin values:", np.unique(spins_plot))
print("fixed spin assignment per electron index?",
      len(np.unique(spins_batch, axis=0)) == 1)

mask_up = spins_plot > 0
mask_dn = ~mask_up
pos_up, pos_dn = positions_plot[mask_up], positions_plot[mask_dn]
print(f"N_up = {mask_up.sum()}, N_dn = {mask_dn.sum()}")

In [ ]:
a1 = np.array(lat_vec[:,0])
a2 = np.array(lat_vec[:,1])
A = np.stack([a1, a2])              # rows are lattice vectors
Ainv = np.linalg.inv(A)
cell_area = abs(np.linalg.det(A))   # 10.392 = (sqrt(3)/2) * a^2

# Cartesian -> fractional:  r = f1*a1 + f2*a2  =>  f = r @ Ainv
frac_plot = positions_plot @ Ainv
print("fractional range:", frac_plot.min(axis=0), frac_plot.max(axis=0))

In [ ]:
nbins = 60                        # ~40 electrons/bin here; see note below
e1 = np.linspace(-0.5, 0.5, nbins + 1)
e2 = np.linspace(-0.5, 0.5, nbins + 1)

f_up = pos_up @ Ainv
f_dn = pos_dn @ Ainv

H_up, _, _ = np.histogram2d(f_up[:, 0], f_up[:, 1], bins=[e1, e2])
H_dn, _, _ = np.histogram2d(f_dn[:, 0], f_dn[:, 1], bins=[e1, e2])

bin_area = cell_area * (e1[1] - e1[0]) * (e2[1] - e2[0])
norm = n_conf * bin_area
n_tot = (H_up + H_dn) / norm
s_z   = (H_up - H_dn) / norm

# fractional bin-edge grid -> Cartesian quad corners
F1, F2 = np.meshgrid(e1, e2, indexing="ij")
X = F1 * a1[0] + F2 * a2[0]
Y = F1 * a1[1] + F2 * a2[1]

print("check normalization:", n_tot.sum() * bin_area, "should be ~", nelec)

In [ ]:
# unit-cell outline for the scatter panels
corners = np.array([[-.5,-.5],[.5,-.5],[.5,.5],[-.5,.5],[-.5,-.5]]) @ A

fig, axes = plt.subplots(2, 2, figsize=(11, 10))

for ax, pos, c, t in zip(axes[0], [pos_up, pos_dn], ["tab:red", "tab:blue"],
                         [r"$n_\uparrow$", r"$n_\downarrow$"]):
    ax.scatter(pos[:, 0], pos[:, 1], color=c, s=0.0005, alpha=0.2)
    ax.plot(corners[:, 0], corners[:, 1], "k-", lw=0.8)
    ax.set_title(t)

im0 = axes[1, 0].pcolormesh(X, Y, n_tot, cmap="viridis", shading="flat")
axes[1, 0].set_title(r"charge density  $n_\uparrow+n_\downarrow$")
fig.colorbar(im0, ax=axes[1, 0], fraction=0.046)

vmax = np.abs(s_z).max()
im1 = axes[1, 1].pcolormesh(X, Y, s_z, cmap="bwr", vmin=-vmax, vmax=vmax,
                            shading="flat")
axes[1, 1].set_title(r"spin density  $n_\uparrow-n_\downarrow$")
fig.colorbar(im1, ax=axes[1, 1], fraction=0.046)

for ax in axes.ravel():
    ax.set_aspect("equal")
    ax.set_xlabel("x"); ax.set_ylabel("y")

plt.tight_layout()
plt.show()

In [ ]:
M = A @ np.linalg.inv(Bmat)
print(np.round(M, 6))        # [[-2, 4], [4, -2]]
print(abs(np.linalg.det(M))) # 12

In [ ]:
Mi = np.rint(M).astype(int)
nsite = int(round(abs(np.linalg.det(Mi))))

pq = np.array([[p, q] for p in range(-12, 13) for q in range(-12, 13)])
f = pq @ np.linalg.inv(Mi)
f -= np.floor(f + 0.5 + 1e-9)          # [-0.5, 0.5), boundary -> -0.5
_, idx = np.unique(np.round(f, 6), axis=0, return_index=True)
sites = f[idx] @ A
print("n sites:", len(sites), "expected:", nsite)

In [ ]:
def site_index(pos, sites, A, Ainv):
    d = pos[:, None, :] - sites[None, :, :]
    df = d @ Ainv
    df -= np.round(df)                  # minimum image
    d = df @ A
    return np.argmin((d**2).sum(-1), axis=1)

site_of = site_index(positions_plot, sites, A, Ainv).reshape(n_conf, nelec)

nsite = len(sites)
sz_conf = np.stack([np.where(site_of == k, spins_batch, 0).sum(1)
                    for k in range(nsite)], axis=1)   # (n_conf, nsite)
n_conf_site = np.stack([(site_of == k).sum(1) for k in range(nsite)], axis=1)

sz   = sz_conf.mean(0)
err  = sz_conf.std(0) / np.sqrt(n_conf)       # optimistic: ignores walker correlation
occ  = n_conf_site.mean(0)

for k in range(nsite):
    print(f"site {k}  r={sites[k].round(2)}  n={occ[k]:.2f}  Sz={sz[k]:+.3f} ± {err[k]:.3f}")
print("sum Sz:", sz.sum())

In [ ]:
for k in range(nsite):
    m = n_conf_site[:, k] == 3
    print(f"site {k}: P(n=3)={m.mean():.3f}  <Sz|n=3>={sz_conf[m,k].mean():+.4f}  "
          f"std={sz_conf[m,k].std():.3f}")

In [ ]:
Grec = 2*np.pi*np.linalg.inv(A).T
Brec = 2*np.pi*np.linalg.inv(Bmat).T
qs = np.array([[i,j] for i in range(-6,7) for j in range(-6,7)]) @ Grec
fq = qs @ np.linalg.inv(Brec); fq -= np.floor(fq + 0.5 + 1e-9)
_, idx = np.unique(np.round(fq,6), axis=0, return_index=True)
qs = fq[idx] @ Brec

Sq = np.abs(np.exp(1j * sites @ qs.T).T @ sz)**2 / nsite
o = np.argsort(Sq)[::-1]
for i in o[:4]:
    print(f"q={qs[i].round(3)}  frac={(qs[i] @ np.linalg.inv(Brec)).round(3)}  S(q)={Sq[i]:.2f}")

In [ ]:
n_ckpt = shape_pos[0]
per_ckpt = n_conf // n_ckpt
sz_ck = sz_conf.reshape(n_ckpt, per_ckpt, nsite).mean(1)   # (n_ckpt, nsite)

print("between-ckpt std per site:", sz_ck.std(0).round(4))
print("within-ckpt SE per site:  ", (sz_conf.std(0)/np.sqrt(per_ckpt)).round(4))
np.set_printoptions(precision=2, suppress=True)
print(sz_ck)

In [ ]:
#Lattice vectors
a1 = np.array(lat_vec[:,0])
a2 = np.array(lat_vec[:,1])

In [ ]:
# Primitive reciprocal lattice vectors
b1 = (2 * np.pi / area) * np.array([a2[1], -a2[0]])
b2 = (2 * np.pi / area) * np.array([-a1[1], a1[0]])

In [ ]:
# --- 3. Convert Cartesian to Fractional Coordinates ---
# Matrix inverse approach: s = R^(-1) * r
lat_vec_inv = np.linalg.inv(lat_vec)
# Vectorized multiplication for all positions
fractional_positions = positions_plot @ lat_vec_inv.T
# Enforce periodic boundaries just in case any drifted out
#fractional_positions = positions_plot % 1.0

In [ ]:
# --- 4. Setup Fourier Grid and Cutoff ---
# Choose grid size for FFT (must be large enough to hold G_max)
# N_grid x N_grid defines the maximum frequencies tracked
N_grid = 128  
G_max = np.linalg.norm(8*b1)  # Cutoff radius for smoothing. Lower = smoother, Higher = sharper.

# Create structure factor grid
rho_G = np.zeros((N_grid, N_grid), dtype=complex)

# Generate frequencies mapping to standard FFT layout
# (0 to N/2, then -N/2 to -1)
n1_range = np.fft.fftfreq(N_grid, d=1/N_grid) 
n2_range = np.fft.fftfreq(N_grid, d=1/N_grid) 

s1 = fractional_positions[:, 0]
s2 = fractional_positions[:, 1]

In [ ]:
# --- 5. Accumulate rho(G) ---
# Loop over the grid indices
for i, n1 in enumerate(n1_range):
    for j, n2 in enumerate(n2_range):
        # Calculate actual G-vector to check the physical circular cutoff
        G_vec = n1 * b1 + n2 * b2
        G_norm = np.linalg.norm(G_vec)
        
        if G_norm <= G_max:
            # G . r = 2 * pi * (n1*s1 + n2*s2)
            phases = 2 * np.pi * (n1 * s1 + n2 * s2)
            # Average over batches (divide by N_batch, not total_points, so rho(G=0) = Ne)
            rho_G[i, j] = np.sum(np.exp(-1j * phases)) / (batch_size*load_N_ckpts)

In [ ]:
# --- 6. Inverse FFT back to Real Space ---
# ifft2 expects the layout we set up. 
# Multiplying by (N_grid^2 / A) handles the proper continuous density scaling
rho_real_fractional = np.fft.ifft2(rho_G) * (N_grid**2 / area)
rho_real = np.real(rho_real_fractional) # Drop microscopic imaginary noise

# --- 7. Plotting the Parallelogram Density ---
# Generate a uniform grid in fractional space
s1_g, s2_g = np.meshgrid(np.linspace(0, 1, N_grid), np.linspace(0, 1, N_grid), indexing='ij')

# Transform the uniform evaluation grid back to Cartesian coordinates for plotting
X_grid = s1_g * a1[0] + s2_g * a2[0]
Y_grid = s1_g * a1[1] + s2_g * a2[1]

In [ ]:
# Use pcolormesh because it handles non-orthogonal, warped grids perfectly
plt.figure(figsize=(6, 5))
plt.pcolormesh(X_grid, Y_grid, rho_real, shading='auto', cmap='viridis') 
plt.colorbar(label='Electron Density $\\rho(\\mathbf{r})$')
#plt.title('Smooth Electron Density (Reciprocal Space Filtered)')
#plt.scatter(X_grid[3, 0]+0.5,Y_grid[3, 0]+np.sqrt(3)/2, color = 'red')
#plt.scatter(0.5,np.sqrt(3)/2)
plt.xlabel('x')
plt.ylabel('y')
plt.axis('equal')
#plt.savefig("phi20_aM98nm_Sz0_den.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
shape_spins = np.shape(spins_ckpt)
# Flattens (30, 2, 512, 12) -> (30720, 12)
spins_batch = np.reshape(np.array(spins_ckpt), (shape_spins[0] * shape_spins[1] * shape_spins[2], num_electrons))

In [ ]:
# Create boolean masks based on the spin values
spin_up_mask = (spins_batch == 1.)
spin_down_mask = (spins_batch == -1.)

# Apply the masks to your unit-cell corrected positions
# This automatically flattens the arrays to (N_up_total, 2) and (N_down_total, 2)
positions_up = positions_batch[spin_up_mask]
positions_down = positions_batch[spin_down_mask]

In [ ]:
fractional_positions_up = positions_up @ lat_vec_inv.T
fractional_positions_down = positions_down @ lat_vec_inv.T

In [ ]:
# Create structure factor grid
rho_G_up = np.zeros((N_grid, N_grid), dtype=complex)
rho_G_down = np.zeros((N_grid, N_grid), dtype=complex)
# Generate frequencies mapping to standard FFT layout
# (0 to N/2, then -N/2 to -1)
#n1_range = np.fft.fftfreq(N_grid, d=1/N_grid) 
#n2_range = np.fft.fftfreq(N_grid, d=1/N_grid) 

s1_up = fractional_positions_up[:, 0]
s2_up = fractional_positions_up[:, 1]
s1_down = fractional_positions_down[:, 0]
s2_down = fractional_positions_down[:, 1]

In [ ]:
# --- 5. Accumulate rho(G) ---
# Loop over the grid indices
for i, n1 in enumerate(n1_range):
    for j, n2 in enumerate(n2_range):
        # Calculate actual G-vector to check the physical circular cutoff
        G_vec = n1 * b1 + n2 * b2
        G_norm = np.linalg.norm(G_vec)
        
        if G_norm <= G_max:
            # G . r = 2 * pi * (n1*s1 + n2*s2)
            phases_up = 2 * np.pi * (n1 * s1_up + n2 * s2_up)
            phases_down = 2 * np.pi * (n1 * s1_down + n2 * s2_down)
            # Average over batches (divide by N_batch, not total_points, so rho(G=0) = Ne)
            rho_G_up[i, j] = np.sum(np.exp(-1j * phases_up)) / (batch_size*load_N_ckpts)
            rho_G_down[i, j] = np.sum(np.exp(-1j * phases_down)) / (batch_size*load_N_ckpts)

In [ ]:
# --- 6. Inverse FFT back to Real Space ---
# ifft2 expects the layout we set up. 
# Multiplying by (N_grid^2 / A) handles the proper continuous density scaling
rho_real_fractional_up = np.fft.ifft2(rho_G_up) * (N_grid**2 / area)
rho_real_up = np.real(rho_real_fractional_up) # Drop microscopic imaginary noise

# --- 7. Plotting the Parallelogram Density ---
# Generate a uniform grid in fractional space
s1_g_up, s2_g_up = np.meshgrid(np.linspace(0, 1, N_grid), np.linspace(0, 1, N_grid), indexing='ij')

# Transform the uniform evaluation grid back to Cartesian coordinates for plotting
X_grid_up = s1_g_up * a1[0] + s2_g_up * a2[0]
Y_grid_up = s1_g_up * a1[1] + s2_g_up * a2[1]

rho_real_fractional_down = np.fft.ifft2(rho_G_down) * (N_grid**2 / area)
rho_real_down = np.real(rho_real_fractional_down) # Drop microscopic imaginary noise

# --- 7. Plotting the Parallelogram Density ---
# Generate a uniform grid in fractional space
s1_g_down, s2_g_down = np.meshgrid(np.linspace(0, 1, N_grid), np.linspace(0, 1, N_grid), indexing='ij')

# Transform the uniform evaluation grid back to Cartesian coordinates for plotting
X_grid_down = s1_g_down * a1[0] + s2_g_down * a2[0]
Y_grid_down = s1_g_down * a1[1] + s2_g_down * a2[1]

In [ ]:
# Use pcolormesh because it handles non-orthogonal, warped grids perfectly
plt.figure(figsize=(6, 5))
plt.pcolormesh(X_grid_up, Y_grid_up, rho_real_up, shading='auto', cmap='Blues')
plt.colorbar(label='$\\rho_u(\\mathbf{r})$')
#plt.title('Smooth Electron Density (Reciprocal Space Filtered)')
plt.xlabel('x')
plt.ylabel('y')
plt.axis('equal')
#plt.savefig("phi45_aM10nm_Sz0_den.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Use pcolormesh because it handles non-orthogonal, warped grids perfectly
plt.figure(figsize=(6, 5))
plt.pcolormesh(X_grid_down, Y_grid_down, rho_real_down, shading='auto', cmap='Blues')
plt.colorbar(label='$\\rho_d(\\mathbf{r})$')
#plt.title('Smooth Electron Density (Reciprocal Space Filtered)')
plt.xlabel('x')
plt.ylabel('y')
plt.axis('equal')
#plt.savefig("phi45_aM10nm_Sz0_den.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# The physical spin density map
spin_density = (rho_real_up - rho_real_down)

# Plotting
fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(10, 4))
# A diverging colormap (like RdBu) is best for spin density so zero is white, up is red, down is blue
ax1.pcolormesh(X_grid, Y_grid, rho_real, shading='auto', cmap='viridis')
ax2.pcolormesh(X_grid, Y_grid, spin_density, shading='auto', cmap='RdBu_r', vmin=-np.max(np.abs(spin_density)), vmax=np.max(np.abs(spin_density)))

#ax2.colorbar(label='Spin Density $s(\\mathbf{r})$')
#plt.title('Smooth Spin Density')
ax1.axis('equal')
ax2.axis('equal')
#plt.savefig("phi20_aM98nm_spin_den_Sz0.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
l = np.linalg.norm(a1)

In [ ]:
from matplotlib.patches import Polygon

#a1 = np.array([2.0, 0.0])
#a2 = np.array([-1.0, np.sqrt(3)])

base_corners = np.array([
    [0, 0],
    a1,
    a1 + a2,
    a2
])
cw = 1.0
def plot_periodic(ax, X_grid, Y_grid, data, cmap, n_repeat=2,
                   xlim=(-cw+np.min([a1[0],a2[0]]), cw+np.max([a1[0],a2[0]])), ylim=(-cw+np.min([a1[1],a2[1]]), cw+np.max([a1[0],a2[0]])), title=None, label_fontsize=14, tick_fontsize=12,panel_label=None, panel_label_fontsize=18):
    data_min, data_max = data.min(), data.max()

    if data_min < -3 and data_max > 0:
        # diverging data: symmetric limits centered at zero
        vabs = max(abs(data_min), abs(data_max))
        vmin, vmax = -vabs, vabs
    else:
        # single-signed data: use actual range
        vmin, vmax = data_min, data_max
    mesh = None
    for i in range(-n_repeat, n_repeat + 1):
        for j in range(-n_repeat, n_repeat + 1):
            shift = i * a1 + j * a2
            m = ax.pcolormesh(X_grid + shift[0], Y_grid + shift[1], data,
                               shading='auto', cmap=cmap,
                               vmin=vmin, vmax=vmax, rasterized=True)
            if mesh is None:
                mesh = m

    outline = Polygon(base_corners, closed=True, fill=False,
                       edgecolor='red', linewidth=2, zorder=10)
    ax.add_patch(outline)

    ax.set_aspect('equal')
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel(r'$x$ [$a_M$]', fontsize=label_fontsize)
    ax.set_ylabel(r'$y$ [$a_M$]', fontsize=label_fontsize)
    ax.tick_params(axis='both', labelsize=tick_fontsize)
    if title:
        ax.set_title(title)
    if panel_label:
        ax.text(-0.15, 1.0, panel_label, transform=ax.transAxes,
         fontsize=panel_label_fontsize, fontweight='bold',
         va='top', ha='right')    
    return mesh

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
label_fontsize = 18
tick_fontsize = 16
cbar_label_fontsize = 18
cbar_tick_fontsize = 16

mesh1 = plot_periodic(axes[0], X_grid, Y_grid, rho_real, cmap='viridis',
                       title=None,
                       label_fontsize=label_fontsize, tick_fontsize=tick_fontsize,panel_label='(a)')
mesh2 = plot_periodic(axes[1], X_grid, Y_grid, spin_density, cmap='RdBu_r',
                       title=None,
                       label_fontsize=label_fontsize, tick_fontsize=tick_fontsize,panel_label='(b)')

cbar1 = fig.colorbar(mesh1, ax=axes[0], label=r'$\rho$', fraction=0.046, pad=0.04)
cbar1.set_label(r'$\rho$', fontsize=cbar_label_fontsize)
cbar1.ax.tick_params(labelsize=cbar_tick_fontsize)

cbar2 = fig.colorbar(mesh2, ax=axes[1], label=r'$\rho_s$', fraction=0.046, pad=0.04)
cbar2.set_label(r'$\rho_s$', fontsize=cbar_label_fontsize)
cbar2.ax.tick_params(labelsize=cbar_tick_fontsize)

plt.tight_layout()
plt.savefig("charge_spin_36.pdf", dpi=150, bbox_inches="tight")
plt.show()